In [3]:
from datasets import load_dataset, concatenate_datasets, DatasetDict
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import time
import evaluate
import pandas as pd
import numpy as np
import os
import wandb

In [4]:
huggingface_dataset_name = "tourmii/vietnamese-corrector-errors"

dataset = load_dataset(huggingface_dataset_name)

dataset

DatasetDict({
    train: Dataset({
        features: ['noisy', 'gt', 'type'],
        num_rows: 15000000
    })
    test: Dataset({
        features: ['noisy', 'gt', 'type'],
        num_rows: 500000
    })
})

In [5]:
model_name='google-t5/t5-base'

original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16).to('cuda') 
full_tuned_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16).to('cuda') 
# original_peft_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16).to('cuda') 

tokenizer = AutoTokenizer.from_pretrained(model_name)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 257/257 [00:00<00:00, 4365.76it/s]


In [6]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

print(print_number_of_trainable_model_parameters(original_model))

trainable model parameters: 222903552
all model parameters: 222903552
percentage of trainable model parameters: 100.00%


In [7]:
index = 30

sentence = dataset['test'][index]['noisy']
correction = dataset['test'][index]['gt']

prompt = f"""
Correct the grammatical mistakes in the following sentence.

{sentence}

Correction:
"""

inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
output = tokenizer.decode(
    original_model.generate(
        inputs["input_ids"], 
        max_new_tokens=200,
    )[0], 
    skip_special_tokens=True
)

dash_line = '-'.join('' for x in range(80))
print(dash_line)
print(f'INPUT PROMPT:\n{prompt}')
print(dash_line)
print(f'BASELINE HUMAN CORRECTION:\n{correction}\n')
print(dash_line)
print(f'MODEL GENERATION - ZERO SHOT:\n{output}')

-------------------------------------------------------------------------------
INPUT PROMPT:

Correct the grammatical mistakes in the following sentence.

Hoc trong nuoc, lay bang tien sy kinh te NorthCentral.

Correction:

-------------------------------------------------------------------------------
BASELINE HUMAN CORRECTION:
Học trong nước, lấy bằng tiến sỹ kinh tế NorthCentral.

-------------------------------------------------------------------------------
MODEL GENERATION - ZERO SHOT:
True


In [ ]:
def tokenize_function(example):
    start_prompt = 'Correct the grammatical errors in the following sentence.\n\n'
    end_prompt = '\n\nCorrection: '
    prompts = [start_prompt + sentence + end_prompt for sentence in example["noisy"]]
    corrections = example["gt"]
    example['input_ids'] = tokenizer(prompts, padding="max_length", truncation=True, max_length=100, return_tensors="pt").input_ids.to('cuda')
    example['labels'] = tokenizer(corrections, padding="max_length", max_length=100, truncation=True, return_tensors="pt").input_ids.to('cuda')
    return example

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(['noisy', 'gt'])

Map:   5%|▌         | 762000/15000000 [01:07<20:54, 11345.38 examples/s]


KeyboardInterrupt: 